# Lesson 6 | What is RTL?

Boolean relationships are now familiar. We need to describe inputs, outputs, stored state, and update timing. Today asks:
> **How can code-like text describe digital hardware rather than a program for a CPU to execute line by line?**

Primary concept: **Register-Transfer Level (RTL)**.


## 1. Concept ledger

**Known:** Boolean logic, register, clock, and clock edge.

**New:** RTL, HDL, SystemVerilog, module, and port.

**Preview only:** `always_ff` appears once so you can read the example; next lesson combines combinational and sequential RTL. Testbenches and waveforms wait until Lesson 8.


## 2. Four terms

**Hardware Description Language (HDL):** a language class for describing digital hardware structure and behavior.

**SystemVerilog:** the HDL and verification language used in this project.

**RTL:** a level focused on what registers hold and how data is computed/transferred across clock cycles.

**module / port:** a module is a bounded hardware unit; ports are signals crossing that boundary.


## 3. HDL may look like software while meaning something different

Adjacent Python statements usually imply execution order. RTL often describes hardware relationships existing at the same time. First ask: what are the inputs, outputs, stored state, and update edge?


## 4. Read one already-known behavior: a clocked accumulator

It implements the same rule as the Lesson 4 accumulator, now in SystemVerilog:

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```


## 5. First SystemVerilog syntax

`input` / `output` declare port direction; `logic signed [7:0]` is an 8-bit signed signal.

`always_ff @(posedge clk)` says this block describes register behavior updated on the rising edge. Today you only need to read it.

`<=` is a **nonblocking assignment**, commonly used in sequential RTL. For now, read it as the new value to store at this edge.

One more fact is only marked here, not taught in depth: both `state` and `input_value` are 8-bit signed values, so `state + input_value` wraps as a finite-width bit pattern when the mathematical result leaves `-128..127`. Treat that as a known limitation of this teaching artifact, not as the formal numeric policy. Lesson 7 observes it in an optional experiment; the formal overflow policy remains a later RMD-002 / RMD-003 decision.


## 6. Reset is part of the contract

The `_n` suffix in `rst_n` indicates active-low reset. This example uses synchronous reset, so reset changes state only at a clock edge. Polarity and timing must be explicit.


## 7. Run: compile check only, without teaching simulation early

If Icarus Verilog is installed, the next cell only checks that the SystemVerilog compiles/elaborates. It does **not** run a testbench or prove behavior; those concepts are reserved for Lesson 8.


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
if not iverilog:
    print('Icarus Verilog not found; RTL compile check did not run.')
else:
    subprocess.run([iverilog, '-g2012', '-tnull', str(root/'rtl/learning/clocked_accumulator.sv')], check=True)
    print('COMPILE PASS: clocked_accumulator.sv')


## 8. Observe

`COMPILE PASS` establishes syntax/basic elaboration only. It does not prove inputs `1,2,3` produce states `1,3,6`. That gap motivates the later testbench lesson.


## 9. Try It

Without simulation, identify two input ports, the output port, stored state, the clock edge, and the reset branch in the code.


## Exercise

[Lesson 6 exercise: translate one RTL clock edge into Python semantics](../../exercises/en/06_what_is_rtl.ipynb)

The standalone workbook checks this lesson's semantic understanding; the real RTL/testbench experiment in the lesson remains part of the course.

## 10. AI Task

Ask AI only to annotate the existing module: boundary, ports, stored state, and clocked update. Do not let it redesign the code.


## 11. Human Check

Without AI, explain HDL vs ordinary software semantics, module vs port, what `always_ff @(posedge clk)` tells you, and why compile success is not behavior verification.


## 12. Engineering Handoff

`rtl/learning/clocked_accumulator.sv` is a teaching artifact. Next lesson decomposes a known neuron contract into a combinational path plus sequential register update.


## 13. Project Trace

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- Teaching RTL: `rtl/learning/clocked_accumulator.sv`


## 14. Exit Ticket

You can explain RTL / HDL / SystemVerilog / module / port, read the state and update edge in a minimal clocked module, and avoid confusing “compiled” with “verified.”
